# H100 Profiling - One Model at a Time

Run each model cell individually to avoid disk space and memory issues.

**Benefits:**
- ✅ Run models in any order
- ✅ Stop anytime, resume later
- ✅ Automatic cleanup after each model
- ✅ Results saved continuously
- ✅ No disk space issues (clears cache)

---

## Setup (Run Once)

In [ ]:
# Check GPU
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    x = torch.ones(100, 100, device='cuda')
    print("✓ GPU works!")
    del x
    torch.cuda.empty_cache()

In [ ]:
# Install dependencies
!pip install -q transformers diffusers accelerate safetensors huggingface-hub

In [ ]:
# Import model runner
import sys
import os
os.chdir('/workspace/h100_profiling')
sys.path.insert(0, '/workspace/h100_profiling')

from run_model import MODEL_RUNNERS
print(f"✓ Loaded {len(MODEL_RUNNERS)} models")

In [ ]:
# Setup functions
import torch
import torch.profiler as profiler
import gc
import json
import time
import os
import shutil
from datetime import datetime

os.makedirs("/workspace/results", exist_ok=True)

# Load existing results
try:
    with open('/workspace/results/summary.json', 'r') as f:
        results_summary = json.load(f)
    print(f"Loaded {len(results_summary)} existing results")
except FileNotFoundError:
    results_summary = []
    print("Starting fresh")

BATCHES = [1, 2, 4, 8, 16, 32, 64, 128]

def cleanup_model():
    """Free GPU + disk space"""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    
    # Clear HuggingFace cache
    cache_dirs = ["/workspace/.cache/huggingface/hub", 
                  os.path.expanduser("~/.cache/huggingface/hub")]
    for cache_dir in cache_dirs:
        if os.path.exists(cache_dir):
            try:
                shutil.rmtree(cache_dir)
                os.makedirs(cache_dir)
            except:
                pass
    print("  🧹 Cleaned")

def save_results():
    with open('/workspace/results/summary.json', 'w') as f:
        json.dump(results_summary, f, indent=2)
    print(f"  💾 Saved {len(results_summary)} results")

def profile_model(model_name, batch_size, output_dir="/workspace/results"):
    """Profile one model+batch"""
    from run_model import MODEL_RUNNERS
    
    if model_name not in MODEL_RUNNERS:
        raise ValueError(f"Unknown model: {model_name}")
    
    runner = MODEL_RUNNERS[model_name]
    
    # Warmup
    try:
        runner(batch_size)
        torch.cuda.synchronize()
    except torch.cuda.OutOfMemoryError:
        return None
    
    torch.cuda.empty_cache()
    
    # Profile
    with profiler.profile(
        activities=[profiler.ProfilerActivity.CUDA],
        record_shapes=True,
        profile_memory=True,
    ) as prof:
        runner(batch_size)
        torch.cuda.synchronize()
    
    # Save trace
    trace_file = f"{output_dir}/trace_{model_name}_b{batch_size}.json"
    prof.export_chrome_trace(trace_file)
    
    # Extract stats
    events = prof.key_averages()
    results = []
    
    for evt in events:
        if evt.device_type == profiler.DeviceType.CUDA:
            kernel_name = evt.key.lower()
            if any(x in kernel_name for x in ['attention', 'attn', 'qkv', 'softmax']):
                op_class = 'attention'
            elif any(x in kernel_name for x in ['gemm', 'conv', 'matmul', 'linear', 'mlp']):
                op_class = 'dense'
            else:
                op_class = 'other'
            
            # Handle different PyTorch versions
            if hasattr(evt, 'cuda_time_total'):
                cuda_time = evt.cuda_time_total
            elif hasattr(evt, 'device_time_total'):
                cuda_time = evt.device_time_total
            else:
                cuda_time = evt.self_cuda_time_total
            
            results.append({
                'kernel_name': evt.key,
                'op_class': op_class,
                'cuda_time_us': cuda_time,
                'count': evt.count,
            })
    
    # Save kernels
    kernels_file = f"{output_dir}/kernels_{model_name}_b{batch_size}.json"
    with open(kernels_file, 'w') as f:
        json.dump({'model': model_name, 'batch': batch_size, 'kernels': results}, f, indent=2)
    
    # Summary
    total = sum(r['cuda_time_us'] for r in results)
    attn = sum(r['cuda_time_us'] for r in results if r['op_class'] == 'attention')
    dense = sum(r['cuda_time_us'] for r in results if r['op_class'] == 'dense')
    
    return {
        'total_time_ms': total / 1000,
        'attention_time_ms': attn / 1000,
        'dense_time_ms': dense / 1000,
        'attention_pct': 100 * attn / total if total > 0 else 0,
        'dense_pct': 100 * dense / total if total > 0 else 0,
    }

print("✓ Setup complete")

In [ ]:
# Fix DiT-XL VAE loading
import run_model
from diffusers import DiTPipeline, AutoencoderKL

def run_dit_xl_fixed(batch_size):
    vae = AutoencoderKL.from_pretrained(
        "stabilityai/sd-vae-ft-msa",
        torch_dtype=torch.bfloat16
    ).to("cuda")
    
    pipe = DiTPipeline.from_pretrained(
        "facebook/DiT-XL-2-256",
        vae=vae,
        torch_dtype=torch.bfloat16
    ).to("cuda")
    
    with torch.no_grad():
        images = pipe(batch_size=batch_size, num_inference_steps=20).images
    torch.cuda.synchronize()
    return images

run_model.MODEL_RUNNERS['dit-xl'] = lambda b: run_dit_xl_fixed(b)
print("✓ DiT-XL fixed")

---
## Models (Run One at a Time)

Each cell profiles one model across all batch sizes.

In [ ]:
# Model 1: ResNet-50
model_name = "resnet-50"
print(f"\n{'='*60}\n{model_name}\n{'='*60}")

for batch in BATCHES:
    print(f"Batch {batch}... ", end="", flush=True)
    try:
        result = profile_model(model_name, batch)
        if result:
            results_summary.append({'model': model_name, 'batch': batch, 'status': 'success', **result})
            print(f"✓ {result['total_time_ms']:.1f}ms")
        else:
            results_summary.append({'model': model_name, 'batch': batch, 'status': 'failed'})
            print("✗ OOM")
            break
    except Exception as e:
        print(f"✗ {str(e)[:50]}")
        break
    cleanup_model()

save_results()
print(f"✓ {model_name} complete\n")

In [ ]:
# Model 2: UNet-SD
model_name = "unet-sd"
print(f"\n{'='*60}\n{model_name}\n{'='*60}")

for batch in BATCHES:
    print(f"Batch {batch}... ", end="", flush=True)
    try:
        result = profile_model(model_name, batch)
        if result:
            results_summary.append({'model': model_name, 'batch': batch, 'status': 'success', **result})
            print(f"✓ {result['total_time_ms']:.1f}ms")
        else:
            results_summary.append({'model': model_name, 'batch': batch, 'status': 'failed'})
            print("✗ OOM")
            break
    except Exception as e:
        print(f"✗ {str(e)[:50]}")
        break
    cleanup_model()

save_results()
print(f"✓ {model_name} complete\n")

In [ ]:
# Model 3: SDXL
model_name = "sdxl"
print(f"\n{'='*60}\n{model_name}\n{'='*60}")

for batch in BATCHES:
    print(f"Batch {batch}... ", end="", flush=True)
    try:
        result = profile_model(model_name, batch)
        if result:
            results_summary.append({'model': model_name, 'batch': batch, 'status': 'success', **result})
            print(f"✓ {result['total_time_ms']:.1f}ms")
        else:
            results_summary.append({'model': model_name, 'batch': batch, 'status': 'failed'})
            print("✗ OOM")
            break
    except Exception as e:
        print(f"✗ {str(e)[:50]}")
        break
    cleanup_model()

save_results()
print(f"✓ {model_name} complete\n")

In [ ]:
# Model 4: DiT-XL
model_name = "dit-xl"
print(f"\n{'='*60}\n{model_name}\n{'='*60}")

for batch in BATCHES:
    print(f"Batch {batch}... ", end="", flush=True)
    try:
        result = profile_model(model_name, batch)
        if result:
            results_summary.append({'model': model_name, 'batch': batch, 'status': 'success', **result})
            print(f"✓ {result['total_time_ms']:.1f}ms")
        else:
            results_summary.append({'model': model_name, 'batch': batch, 'status': 'failed'})
            print("✗ OOM")
            break
    except Exception as e:
        print(f"✗ {str(e)[:50]}")
        break
    cleanup_model()

save_results()
print(f"✓ {model_name} complete\n")

In [ ]:
# Model 5: UNet-3D
model_name = "unet-3d"
print(f"\n{'='*60}\n{model_name}\n{'='*60}")

for batch in BATCHES:
    print(f"Batch {batch}... ", end="", flush=True)
    try:
        result = profile_model(model_name, batch)
        if result:
            results_summary.append({'model': model_name, 'batch': batch, 'status': 'success', **result})
            print(f"✓ {result['total_time_ms']:.1f}ms")
        else:
            results_summary.append({'model': model_name, 'batch': batch, 'status': 'failed'})
            print("✗ OOM")
            break
    except Exception as e:
        print(f"✗ {str(e)[:50]}")
        break
    cleanup_model()

save_results()
print(f"✓ {model_name} complete\n")

In [ ]:
# Model 6-9: Llama variants - add cells for each
# (Cells for llama-8b-1k, llama-8b-4k, llama-8b-16k, llama-8b-64k)
# (Cells for cogvideox variants)
# See pattern above - copy and change model_name

## Final Results

In [ ]:
import pandas as pd

df = pd.DataFrame(results_summary)
print(f"\n{'='*60}")
print(f"FINAL: {len(df)} runs")
print(f"Success: {len(df[df['status']=='success'])}")
print(f"Failed: {len(df[df['status']=='failed'])}")
print(f"{'='*60}\n")
display(df)

!cd /workspace && tar -czf h100_results_final.tar.gz results/
print("\n✓ Download: /workspace/h100_results_final.tar.gz")